Credit Card Fraud Detection

In [60]:
from pathlib import Path
import pandas as pd
from scipy.io import arff

downloads = Path.home() / "Downloads"
print([f.name for f in downloads.glob("*.arff")])

data, _ = arff.loadarff(downloads / "phpKo8OWT.arff")
df = pd.DataFrame(data)
print(df.shape)
print(df.head())

counts = df['Class'].value_counts(dropna=False)
result = counts[b'1'] / len(df) * 100
amount_mean = df['Amount'].mean()
amount_median = df['Amount'].median()

print(counts)
print(df.isna().sum())
print(result)
print("mean:", amount_mean, "median:", amount_median)


['phpKo8OWT.arff']
(284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   



In [61]:
from sklearn.model_selection import train_test_split

x = df.drop(columns='Class')
y = df['Class']
print(x.shape, y.shape)

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.2)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.25, stratify=y_train)

print(x_val.shape[0])
val = y_val.value_counts()

result_val = val[b'1'] / len(y_val) * 100
print(result_val)

train_counts = y_train.value_counts()
result_train = train_counts[b'1'] / len(y_train) * 100
print(result_train)
print(len(y_train))

test_counts = y_test.value_counts()
result_test = test_counts[b'1'] / len(y_test) * 100
print(result_test)
print(len(y_test))


(284807, 30) (284807,)
56962
0.17380007724447877
0.17263273701889598
170883
0.17204452090867595
56962


In [62]:
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [63]:
models = {
    "Baseline: always normal": DummyClassifier(
        strategy="constant", constant=0
    ),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42),
    ),
    "KNN without scaling": KNeighborsClassifier(
        n_neighbors=5, n_jobs=1
    ),
    "KNN with scaling": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}


In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(x_train)
print(round(scaler.mean_[0], 2))

x_train_scaled = scaler.transform(x_train)
print(x_train_scaled[0, :3])

x_val_scaled = scaler.transform(x_val)
print(x_val_scaled.shape)

print(x_train_scaled[:, 0].mean())
print(x_train_scaled[:, 0].std())


94894.62
[-1.21056397  0.6392255   0.15966625]
(56962, 30)
2.6029491089565534e-17
0.9999999999999999


In [65]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int) 

print("Training labels:", y_train.unique())
print("Training label type:", y_train.dtype)

folds = list(cv.split(x_train, y_train))

scoring = {
    "Accuracy": "accuracy",
    "Precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "Recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "F1": make_scorer(f1_score, pos_label=1, zero_division=0),
}

Training labels: [0 1]
Training label type: int64


In [66]:
import joblib
from threadpoolctl import threadpool_limits
from pathlib import Path

SEED = 42
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

In [67]:
from sklearn.utils import resample
import numpy as np

rows = []

x_tr = np.array(x_train, dtype=np.float64)
y_tr = np.array(y_train, dtype=np.int64)

for name, model in models.items():
    print(f"Evaluating: {name}", flush=True)

    if "KNN" in name:
        x_s, y_s = resample(
            x_tr, y_tr,
            n_samples=20_000,
            stratify=y_tr,
            random_state=42
        )
        n_jobs_cv = 1
    else:
        x_s, y_s = x_tr, y_tr
        n_jobs_cv = -1

    scores = cross_validate(
        model, x_s, y_s,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs_cv,
        error_score="raise",
    )

    row = {"Model": name}
    for metric in scoring:
        row[metric] = scores[f"test_{metric}"].mean()
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index("Model")
display(cv_results.round(4))



Evaluating: Baseline: always normal


Evaluating: Logistic Regression
Evaluating: KNN without scaling
Evaluating: KNN with scaling
Evaluating: Decision Tree


,Accuracy,Precision,Recall,F1
Model,,,,
Baseline: always normal,0.9983,0.0000,0.0000,0.0000
Logistic Regression,0.9992,0.8896,0.6305,0.7378
KNN without scaling,0.9982,0.0000,0.0000,0.0000
KNN with scaling,0.9992,0.9000,0.5714,0.6895
Decision Tree,0.9990,0.7130,0.7492,0.7304


In [68]:

knn_results = cv_results.loc[
    ["KNN without scaling", "KNN with scaling"],
    ["Precision", "Recall", "F1"],
]
display(knn_results.round(4))

difference = (
    knn_results.loc["KNN with scaling"]
    - knn_results.loc["KNN without scaling"]
)
print("Change after scaling (percentage points):")

display((100 * difference).round(2))

,Precision,Recall,F1
Model,,,
KNN without scaling,0.0,0.0000,0.0000
KNN with scaling,0.9,0.5714,0.6895


Change after scaling (percentage points):


Precision    90.00
Recall       57.14
F1           68.95
dtype: float64

In [69]:

log_reg = clone(models["Logistic Regression"])
log_reg.fit(x_train, y_train)
y_val_pred = log_reg.predict(x_val)

tn, fp, fn, tp = confusion_matrix(
    y_val, y_val_pred, labels=[0, 1]
).ravel()

print("Normal transactions correctly classified (TN):", tn)
print("Normal transactions incorrectly flagged as fraud (FP):", fp)
print("Fraudulent transactions missed (FN):", fn)
print("Fraudulent transactions detected (TP):", tp)
print("Total fraudulent transactions in the validation set:", fn + tp)

Normal transactions correctly classified (TN): 56848
Normal transactions incorrectly flagged as fraud (FP): 15
Fraudulent transactions missed (FN): 31
Fraudulent transactions detected (TP): 68
Total fraudulent transactions in the validation set: 99


In [70]:
from pathlib import Path
from threadpoolctl import threadpool_limits
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
import pandas as pd

SEED = 42
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

def evaluate(y_true, predictions):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0)
    }

candidates = {"Logistic Regression": clone(models["Logistic Regression"])}

for depth in (3, 5, 8):
    candidates[f"Tree depth {depth}"] = DecisionTreeClassifier(
        max_depth=depth,
        random_state=SEED
    )

trials = []

for name, model in candidates.items():
    with threadpool_limits(limits=1):
        model.fit(x_train, y_train)
        val_scores = model.predict_proba(x_val)[:, 1]


    for threshold in (0.10, 0.25, 0.50, 0.75):
        val_alerts = (val_scores >= threshold).astype(int)

        trials.append({
            "Model": name,
            "Threshold": threshold,
            **evaluate(y_val, val_alerts)
        })

validation_results = pd.DataFrame(trials).sort_values(
    by=["F1", "FP"],
    ascending=[False, True]
)

validation_results.to_csv(OUT / "day6_validation.csv", index=False)
print(validation_results.round(4).to_string(index=False))

best_row = validation_results.iloc[0]
best_name = best_row["Model"]
best_model = candidates[best_name]
best_threshold = float(best_row["Threshold"])

print("\nSelected model:", best_name)
print("Selected threshold:", best_threshold)


 


              Model  Threshold    TN  FP  FN  TP  Accuracy  Precision  Recall     F1
       Tree depth 5       0.50 56851  12  23  76    0.9994     0.8636  0.7677 0.8128
       Tree depth 5       0.10 56849  14  22  77    0.9994     0.8462  0.7778 0.8105
       Tree depth 5       0.25 56849  14  22  77    0.9994     0.8462  0.7778 0.8105
Logistic Regression       0.25 56845  18  21  78    0.9993     0.8125  0.7879 0.8000
Logistic Regression       0.10 56837  26  16  83    0.9993     0.7615  0.8384 0.7981
       Tree depth 5       0.75 56854   9  28  71    0.9994     0.8875  0.7172 0.7933
       Tree depth 8       0.25 56854   9  29  70    0.9993     0.8861  0.7071 0.7865
       Tree depth 8       0.50 56854   9  29  70    0.9993     0.8861  0.7071 0.7865
       Tree depth 8       0.75 56854   9  29  70    0.9993     0.8861  0.7071 0.7865
       Tree depth 3       0.25 56843  20  24  75    0.9992     0.7895  0.7576 0.7732
       Tree depth 3       0.50 56843  20  24  75    0.9992     0.

In [71]:
test_scores = best_model.predict_proba(x_test)[:, 1]
test_alerts = (test_scores >= best_threshold).astype(int)

final_report = evaluate(y_test, test_alerts)

print("Final test results:")
for key, value in final_report.items():
    print(f"{key}: {value}")

pd.DataFrame([{
    "Model": best_name,
    "Threshold": best_threshold,
    **final_report
}]).to_csv(OUT / "day7_final_test.csv", index=False)

bundle = {
    "model": best_model,
    "threshold": best_threshold,
    "features": x_train.columns.tolist()
}

model_path = OUT / "fraud_model.joblib"
joblib.dump(bundle, model_path)
print("Saved model:", model_path)


Final test results:
TN: 56855
FP: 9
FN: 24
TP: 74
Accuracy: 0.999420666409185
Precision: 0.891566265060241
Recall: 0.7551020408163265
F1: 0.8176795580110497
Saved model: outputs/fraud_model.joblib
